# Qiskit BackendV2 quickstart

Build and transpile a Bell circuit, then compare Qiskit's exact state with MettleQ's native BackendV2 result and execution plan.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

A Qiskit backend is the execution target used after transpilation. The circuit itself does not need to be rewritten for MettleQ.

In [2]:
circuit = QuantumCircuit(2, name="bell")
circuit.h(0)
circuit.cx(0, 1)

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(
    lambda: np.asarray(Statevector.from_instruction(circuit).data)
)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(method="statevector", device="auto")
compiled = transpile(circuit, backend, optimization_level=1)

def run_mettleq():
    return np.asarray(
        backend.run(
            compiled,
            shots=1,
            return_statevector=True,
            execution_report=True,
        ).result().data(0)["statevector"]
    )

candidate, mettleq_ms, _ = benchmark(run_mettleq)
error = phase_aligned_statevector_error(reference, candidate)
method, device = qiskit_selection(backend)
plan = backend.last_execution_plans[-1]

## 4. Check correctness before discussing speed

A Bell state has two non-zero amplitudes. We remove an irrelevant global phase before comparing those amplitudes.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/01_backend_quickstart.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="phase-aligned statevector atol=2e-6",
    passed=error <= 2e-6 and compiled.num_qubits == 2,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_amplitude_error": error, "execution_status": plan["execution_status"]},
)


Comparison summary
------------------
Correctness contract: PASS — phase-aligned statevector atol=2e-6
SDK reference median: 0.065 ms
MettleQ median:       0.318 ms
Timing interpretation: the SDK reference was 4.908x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "phase-aligned statevector atol=2e-6", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"execution_status": "evaluated", "max_amplitude_error": 1.2101617041793133e-08}, "mettleq_median_ms": 0.31820801086723804, "notebook": "qiskit/01_backend_quickstart.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 0.06483300239779055, "reference_over_mettleq": 0.2037440924918764, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}


## What should you conclude?

Use this pattern when existing Qiskit code already accepts a BackendV2. This tiny circuit demonstrates integration, not acceleration.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.